# Public data experiments
This is the notebook to run the public data experiment

In [1]:
import sys
from pathlib import Path

# Add the workspace root to sys.path
workspace_root = Path("/Users/eduardohome/workspace/benchmark_credit_scoring")
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

# Now you can import from src
from src.utils.csv_config import read_csv_by_name, create_imbalanced_datasets, merge_xai_results
from src.model.functions import save_model, load_model, get_split_data
from src.xai.functions import get_train_iterations_xai

try:
    from src.model.functions import get_train_model
except Exception as exc:
    # provide helpful message if import fails
    print("Failed to import model utilities:", exc)
    print("This often means a missing dependency such as libomp for LightGBM.")
    print("On macOS you can install it with `brew install libomp` and then restart the kernel.")
    raise

# General imports
import pandas as pd

In [2]:
# -------------------------
# Settings
# -------------------------
data_dir = Path("/Users/eduardohome/workspace/benchmark_credit_scoring/data")
datasets_to_run = ["australian", "german", "japan", "kgccdea", "taiwan", "th02"] #kggmsc
datasets_to_run = ["taiwan", "th02"]
target_col = "target"

# Edit this list whenever you want more/less imbalance datasets
imbalance_levels = [1, 2, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50]

# list of estimators understood by functions.get_model
model_choices = ["RandomForest", "XGBoost", "LightGBM", "BoostDT", "BoostSGB", "BagNN", "AdaBoost-DT", "ANN"] #, "SVM-Rbf"
product = "default"


In [3]:
for dataset_name in datasets_to_run:
    dataset_path = data_dir / "input" / f"{dataset_name}.csv"
    # Check if file exists and read it using the helper from src.csv_config
    if dataset_path.exists():
        # Use the dataset-specific defaults (german uses tab separator)
        create_imbalanced_datasets(
            dataset_name=dataset_name,
            data_dir=data_dir,
            imbalance_levels=imbalance_levels,  # <-- modify freely
            target_col="target",
            )

Original dataset: taiwan
Total samples: 30000
Positive samples: 6636
Negative samples: 23364
Positive rate: 0.2212

taiwan_1.csv | Total=23600 | Pos=236 | Rate=0.0100 | keep all negatives (23364), downsample positives to 236
taiwan_2.csv | Total=23840 | Pos=476 | Rate=0.0200 | keep all negatives (23364), downsample positives to 476
taiwan_5.csv | Total=24593 | Pos=1229 | Rate=0.0500 | keep all negatives (23364), downsample positives to 1229
taiwan_10.csv | Total=25960 | Pos=2596 | Rate=0.1000 | keep all negatives (23364), downsample positives to 2596
taiwan_15.csv | Total=27487 | Pos=4123 | Rate=0.1500 | keep all negatives (23364), downsample positives to 4123
taiwan_20.csv | Total=29205 | Pos=5841 | Rate=0.2000 | keep all negatives (23364), downsample positives to 5841
taiwan_25.csv | Total=26544 | Pos=6636 | Rate=0.2500 | keep all positives (6636), downsample negatives to 19908
taiwan_30.csv | Total=22120 | Pos=6636 | Rate=0.3000 | keep all positives (6636), downsample negatives to 1

In [4]:
datasets_to_run

['taiwan', 'th02']

In [5]:
results = []

for dataset_name in datasets_to_run:
    # variants for this dataset
    dataset_variants = [dataset_name] + [f"{dataset_name}_{pct}" for pct in imbalance_levels]

    # ensure output dir exists
    out_dir = data_dir / "output" / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "#" * 90)
    print(f"BASE DATASET: {dataset_name}")
    print("#" * 90)

    for dataset_variant in dataset_variants:
        dataset_path = data_dir / "intermediate" / dataset_name / f"{dataset_variant}.csv"

        if not dataset_path.exists():
            print(f"[SKIP] File not found: {dataset_path}")
            continue

        # Read each dataset variant (uses your resolver logic in csv_config.py)
        df = pd.read_csv(str(dataset_path))

        # Basic safety checks
        if target_col not in df.columns:
            print(f"[SKIP] {dataset_variant}: target column '{target_col}' not found. Columns={list(df.columns)[:10]}...")
            continue

        # Quick dataset stats
        n_total = len(df)
        # Make sure sum/mean work even if target is int-like
        y = pd.to_numeric(df[target_col], errors="coerce")
        n_pos = int((y == 1).sum())
        pos_rate = float((y == 1).mean())

        print("\n" + "=" * 80)
        print(f"DATASET: {dataset_variant} | n={n_total} | pos={n_pos} | pos_rate={pos_rate:.4f}")
        print("=" * 80)

        # Prepare column lists for this df
        feature_cols     = df.columns.tolist()
        numerical_cols   = df.select_dtypes(include="number").columns.tolist()
        categorical_cols = df.select_dtypes(include="object").columns.tolist()
        ids_cols         = []
        target_cols      = [target_col]
        exclude_cols     = []

        # Optionally skip if only one class present (prevents stratify crash)
        n_neg = n_total - n_pos
        if n_pos == 0 or n_neg == 0:
            print(f"[SKIP] {dataset_variant}: only one class present (pos={n_pos}, neg={n_neg}).")
            continue

        for model_choice in model_choices:
            print(f"\n=== training {model_choice} on {dataset_variant} ===")

            pipeline, best_params, best_score = get_train_model(
                data=df,
                feature_cols=feature_cols,
                numerical_cols=numerical_cols,
                categorical_cols=categorical_cols,
                ids_cols=ids_cols,
                target_cols=target_cols,
                exclude_cols=exclude_cols,
                product=product,
                algorithm=model_choice,
                test_mode=True,  # set False for full search
            )

            print("Best hyperparameters:", best_params)
            print("Best CV score:", best_score)

            model_path = out_dir / f"{dataset_variant}_{model_choice}.pkl"
            save_model(str(model_path), pipeline, environment="local")
            print(f"trained pipeline saved to {model_path}")

            results.append(
                {
                    "base_dataset": dataset_name,
                    "dataset_variant": dataset_variant,
                    "model": model_choice,
                    "n_total": n_total,
                    "n_pos": n_pos,
                    "pos_rate": pos_rate,
                    "params": best_params,
                    "score": best_score,
                }
            )

results_df = pd.DataFrame(results)
results_df


##########################################################################################
BASE DATASET: taiwan
##########################################################################################
[SKIP] File not found: /Users/eduardohome/workspace/benchmark_credit_scoring/data/intermediate/taiwan/taiwan.csv

DATASET: taiwan_1 | n=23600 | pos=236 | pos_rate=0.0100

=== training RandomForest on taiwan_1 ===
Search space for algorithm=RandomForest, product=default, test_mode=True: {'model__n_estimators': Integer(low=50, high=300, prior='uniform', transform='identity'), 'model__max_depth': Integer(low=1, high=50, prior='uniform', transform='identity'), 'model__min_samples_split': Integer(low=2, high=20, prior='uniform', transform='identity'), 'model__min_samples_leaf': Integer(low=1, high=10, prior='uniform', transform='identity'), 'model__max_features': Categorical(categories=('sqrt',), prior=None), 'model__bootstrap': Categorical(categories=(True, False), prior=None), 'model__cla

,base_dataset,dataset_variant,model,n_total,n_pos,pos_rate,params,score
0,taiwan,taiwan_1,RandomForest,23600,236,0.01,"{'model__bootstrap': False, 'model__class_weig...",0.044794
1,taiwan,taiwan_1,XGBoost,23600,236,0.01,"{'model__booster': 'gbtree', 'model__colsample...",0.052920
2,taiwan,taiwan_1,LightGBM,23600,236,0.01,{'model__bagging_fraction': 0.9293468848796316...,0.051039
3,taiwan,taiwan_1,BoostDT,23600,236,0.01,"{'model__learning_rate': 0.07222603429369405, ...",0.040682
4,taiwan,taiwan_1,BoostSGB,23600,236,0.01,"{'model__learning_rate': 0.07222603429369405, ...",0.040436
...,...,...,...,...,...,...,...,...
187,th02,th02_50,BoostDT,646,323,0.50,"{'model__learning_rate': 0.07222603429369405, ...",0.659118
188,th02,th02_50,BoostSGB,646,323,0.50,"{'model__learning_rate': 0.07222603429369405, ...",0.665573
189,th02,th02_50,BagNN,646,323,0.50,"{'model__estimator__activation': 'relu', 'mode...",0.627134
190,th02,th02_50,AdaBoost-DT,646,323,0.50,"{'model__estimator__max_depth': 3, 'model__est...",0.569114


In [6]:
# -------------------------
# Loop XAI iterations
# -------------------------
for dataset_name in datasets_to_run:

    dataset_variants = [dataset_name] + [f"{dataset_name}_{lvl}" for lvl in imbalance_levels]

    out_dir = data_dir / "output" / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "#" * 90)
    print(f"BASE DATASET: {dataset_name}")
    print("#" * 90)

    for dataset_variant in dataset_variants:

        dataset_path = data_dir / "intermediate" / dataset_name / f"{dataset_variant}.csv"

        if not dataset_path.exists():
            print(f"[SKIP] File not found: {dataset_path}")
            continue

        # Same reader used during training
        path = str(dataset_path)
        df = pd.read_csv(str(path))

        if target_col not in df.columns:
            print(f"[SKIP] {dataset_variant}: target column '{target_col}' not found.")
            continue

        n_total = len(df)
        y = pd.to_numeric(df[target_col], errors="coerce")
        n_pos = int((y == 1).sum())
        pos_rate = float((y == 1).mean())

        print("\n" + "=" * 80)
        print(f"DATASET: {dataset_variant} | n={n_total} | pos={n_pos} | pos_rate={pos_rate:.4f}")
        print("=" * 80)

        n_neg = n_total - n_pos
        if n_pos == 0 or n_neg == 0:
            print(f"[SKIP] {dataset_variant}: only one class present.")
            continue

        # Same column preparation as training
        feature_cols     = df.columns.tolist()
        numerical_cols   = df.select_dtypes(include="number").columns.tolist()
        categorical_cols = df.select_dtypes(include="object").columns.tolist()
        ids_cols         = []
        target_cols      = [target_col]
        exclude_cols     = []

        # Data split
        x_train, x_test, y_train, y_test, numerical, categorical = get_split_data(
            data=df,
            feature_cols=feature_cols,
            numerical_cols=numerical_cols,
            categorical_cols=categorical_cols,
            ids_cols=ids_cols,
            target_cols=target_cols,
            exclude_cols=exclude_cols,
            random_state=123,
        )

        for model_choice in model_choices:

            model_path = out_dir / f"{dataset_variant}_{model_choice}.pkl"

            if not model_path.exists():
                print(f"[SKIP] Model not found: {model_path}")
                continue

            print(f"Running XAI iterations for {dataset_variant} | {model_choice}")

            pipeline = load_model(str(model_path))

            (
                df_shap,
                df_ranking_local,
                extra_local,
                df_ranking_global,
                df_resume_local,
                df_resume_global,
                df_resume
            ) = get_train_iterations_xai(
                pipeline,
                x_train,
                x_test,
                y_train,
                n_iterations=5,
                percentage=90,
            )

            # ----------------------------
            # Create output folder
            # ----------------------------
            result_dir = out_dir / f"{dataset_variant}_{model_choice}_xai.csv"
            result_dir.mkdir(parents=True, exist_ok=True)

            # ----------------------------
            # Save outputs
            # ----------------------------
            df_shap.to_csv(
                result_dir / f"{dataset_variant}_{model_choice}_shap_values.csv",
                index=False
            )

            df_ranking_local.to_csv(
                result_dir / f"{dataset_variant}_{model_choice}_ranking_local.csv",
                index=False
            )

            extra_local.to_csv(
                result_dir / f"{dataset_variant}_{model_choice}_ranking_extra_local.csv",
                index=False
            )

            df_ranking_global.to_csv(
                result_dir / f"{dataset_variant}_{model_choice}_ranking_global.csv",
                index=False
            )

            df_resume_local.to_csv(
                result_dir / f"{dataset_variant}_{model_choice}_resume_local.csv",
                index=False
            )

            df_resume_global.to_csv(
                result_dir / f"{dataset_variant}_{model_choice}_resume_global.csv",
                index=False
            )

            df_resume.to_csv(
                result_dir / f"{dataset_variant}_{model_choice}_summary.csv",
                index=False
            )

            print(f"[OK] Saved results to {result_dir}")


##########################################################################################
BASE DATASET: taiwan
##########################################################################################
[SKIP] File not found: /Users/eduardohome/workspace/benchmark_credit_scoring/data/intermediate/taiwan/taiwan.csv

DATASET: taiwan_1 | n=23600 | pos=236 | pos_rate=0.0100
Running XAI iterations for taiwan_1 | RandomForest
Model loaded successfully from local path /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/taiwan/taiwan_1_RandomForest.pkl
[OK] Saved results to /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/taiwan/taiwan_1_RandomForest_xai.csv
Running XAI iterations for taiwan_1 | XGBoost
Model loaded successfully from local path /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/taiwan/taiwan_1_XGBoost.pkl
[OK] Saved results to /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/taiwan/taiwan_1_XGBoost_xai.csv
Running

PermutationExplainer explainer: 7081it [02:12, 49.51it/s]                          
PermutationExplainer explainer: 7081it [02:07, 51.12it/s]                          
PermutationExplainer explainer: 7081it [02:10, 50.17it/s]                          
PermutationExplainer explainer: 7081it [02:07, 51.22it/s]                          
PermutationExplainer explainer: 7081it [02:07, 50.99it/s]                          


[OK] Saved results to /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/taiwan/taiwan_1_BoostDT_xai.csv
Running XAI iterations for taiwan_1 | BoostSGB
Model loaded successfully from local path /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/taiwan/taiwan_1_BoostSGB.pkl


PermutationExplainer explainer: 7081it [02:11, 49.67it/s]                          
PermutationExplainer explainer: 7081it [02:06, 51.54it/s]                          
PermutationExplainer explainer: 7081it [02:09, 50.37it/s]                          
PermutationExplainer explainer: 7081it [02:06, 51.47it/s]                          
PermutationExplainer explainer: 7081it [02:05, 51.89it/s]                          


[OK] Saved results to /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/taiwan/taiwan_1_BoostSGB_xai.csv
Running XAI iterations for taiwan_1 | BagNN
Model loaded successfully from local path /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/taiwan/taiwan_1_BagNN.pkl


...

[OK] Saved results to /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/th02/th02_45_BagNN_xai.csv
Running XAI iterations for th02_45 | AdaBoost-DT
Model loaded successfully from local path /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/th02/th02_45_AdaBoost-DT.pkl
[OK] Saved results to /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/th02/th02_45_AdaBoost-DT_xai.csv
Running XAI iterations for th02_45 | ANN
Model loaded successfully from local path /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/th02/th02_45_ANN.pkl


PermutationExplainer explainer: 217it [00:20,  8.92it/s]                         


[OK] Saved results to /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/th02/th02_45_ANN_xai.csv

DATASET: th02_50 | n=646 | pos=323 | pos_rate=0.5000
Running XAI iterations for th02_50 | RandomForest
Model loaded successfully from local path /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/th02/th02_50_RandomForest.pkl
[OK] Saved results to /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/th02/th02_50_RandomForest_xai.csv
Running XAI iterations for th02_50 | XGBoost
Model loaded successfully from local path /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/th02/th02_50_XGBoost.pkl
[OK] Saved results to /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/th02/th02_50_XGBoost_xai.csv
Running XAI iterations for th02_50 | LightGBM
Model loaded successfully from local path /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/th02/th02_50_LightGBM.pkl
[OK] Saved results to /Users/eduardohome/workspa

PermutationExplainer explainer: 195it [05:10,  3.20s/it]                         


[OK] Saved results to /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/th02/th02_50_BagNN_xai.csv
Running XAI iterations for th02_50 | AdaBoost-DT
Model loaded successfully from local path /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/th02/th02_50_AdaBoost-DT.pkl
[OK] Saved results to /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/th02/th02_50_AdaBoost-DT_xai.csv
Running XAI iterations for th02_50 | ANN
Model loaded successfully from local path /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/th02/th02_50_ANN.pkl
[OK] Saved results to /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/th02/th02_50_ANN_xai.csv


In [7]:
merge_xai_results(
    data_dir=data_dir,
    datasets_to_run=datasets_to_run,
    imbalance_levels=imbalance_levels,
    model_choices=model_choices,
)


Processing dataset: taiwan
Missing file: /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/taiwan/taiwan_1_RandomForest_xai_iterations.csv
Missing file: /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/taiwan/taiwan_1_XGBoost_xai_iterations.csv
Missing file: /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/taiwan/taiwan_1_LightGBM_xai_iterations.csv
Missing file: /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/taiwan/taiwan_1_BoostDT_xai_iterations.csv
Missing file: /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/taiwan/taiwan_1_BoostSGB_xai_iterations.csv
Missing file: /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/taiwan/taiwan_1_BagNN_xai_iterations.csv
Missing file: /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/taiwan/taiwan_1_AdaBoost-DT_xai_iterations.csv
Missing file: /Users/eduardohome/workspace/benchmark_credit_scoring/data/output/taiwan/taiwan_1_ANN_xa